# Cyclical monotonicity as absence of crossings

This notebook generates `fig:kantorovich-cyclical-monotonicity`.  For the quadratic cost, the two-point cyclical monotonicity inequality is
$$
\|x_1-y_1\|^2+\|x_2-y_2\|^2
\leq
\|x_1-y_2\|^2+\|x_2-y_1\|^2.
$$
A crossed assignment violates this inequality in the tiny four-point example below; swapping the two targets gives the optimal uncrossed assignment.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import contextlib
import io

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.colors import to_rgb

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    DIRAC_MARKER_SIZE,
    MASS_MARKER_MIN_FACTOR,
    MASS_MARKER_MAX_FACTOR,
    TRANSPORT_LINE_MIN_WIDTH,
    TRANSPORT_LINE_MAX_WIDTH,
    TRANSPORT_LINE_ALPHA_SCALE,
    draw_point_clouds,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()


def quiet_save_pdf(fig, path, *, pad_inches=0.035):
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        save_pdf(fig, path, pad_inches=pad_inches)

import ot

## A two-by-two assignment

The red source atoms and blue target atoms have equal weights.  We manually draw the crossed permutation and use POT to compute the quadratic optimal assignment, which is uncrossed.

In [ ]:
alpha = np.array([
    [-1.00, -0.45],
    [-1.00,  0.45],
])
beta = np.array([
    [1.00, -0.45],
    [1.00,  0.45],
])
a = np.ones(len(alpha)) / len(alpha)
b = np.ones(len(beta)) / len(beta)

C = ot.dist(alpha, beta, metric="euclidean") ** 2
P_optimal = ot.emd(a, b, C, numItermax=200000)
P_crossed = np.array([
    [0.0, 0.5],
    [0.5, 0.0],
])

cost_optimal = float(np.sum(P_optimal * C))
cost_crossed = float(np.sum(P_crossed * C))
all_points = np.vstack([alpha, beta])

def square_limits(xlim, ylim):
    cx = 0.5 * (xlim[0] + xlim[1])
    cy = 0.5 * (ylim[0] + ylim[1])
    span = max(xlim[1] - xlim[0], ylim[1] - ylim[0])
    return (cx - span / 2, cx + span / 2), (cy - span / 2, cy + span / 2)

xlim, ylim = square_limits(*padded_limits(all_points, pad=0.30))

P_crossed, P_optimal, cost_crossed, cost_optimal

## Assignment panels

Both panels use the same endpoints and the same scale.  Only the assignment changes, so the crossing can be read as a direct local certificate of non-optimality for $c(x,y)=\|x-y\|^2$.

In [ ]:
def assignment_pairs(P):
    return [(i, j, float(P[i, j])) for i in range(P.shape[0]) for j in range(P.shape[1]) if P[i, j] > 1e-12]


def draw_assignment(P, path, *, line_color=VIOLET):
    fig, ax = plt.subplots(figsize=(2.25, 2.25))
    segments = [[alpha[i], beta[j]] for i, j, _ in assignment_pairs(P)]
    base = np.array(to_rgb(line_color))
    ax.add_collection(LineCollection(segments, colors=[(*base, 0.72)], linewidths=1.10, zorder=1))
    draw_point_clouds(ax, alpha, beta, base_size=DIRAC_MARKER_SIZE, zorder=3)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    remove_axes(ax)
    quiet_save_pdf(fig, path, pad_inches=0.085)
    plt.close(fig)

## Exported panels

The panel names and the cost comparison are written in LaTeX, not inside the PDFs.

In [ ]:
fig_name = "kantorovich-cyclical-monotonicity"
out = figure_dir(fig_name)

draw_assignment(P_crossed, out / "crossed.pdf", line_color=ORANGE)
draw_assignment(P_optimal, out / "uncrossed.pdf", line_color=VIOLET)